# EEG-CogState — Comparaison de modèles

**Objectif** : comparer plusieurs algorithmes de classification sur les mêmes données, avec le **même protocole d'évaluation par sujet** (LOSO). Cela permet de choisir le meilleur modèle de manière objective.

Modèles comparés : Random Forest, SVM (RBF), Régression logistique, XGBoost.

**Prérequis** : `features.csv` (Sprint 3).

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src import model, model_comparison

X, y, groups, names = model.load_features()
print(f"{X.shape[0]} epochs, {len(np.unique(groups))} sujets, {X.shape[1]} features")

## 1. Lancer la comparaison

Chaque modèle est entraîné et évalué par validation Leave-One-Subject-Out.

> Peut prendre une à quelques minutes selon le nombre de sujets et de modèles.

In [ ]:
results = model_comparison.compare_models(X, y, groups, verbose=True)
print()
results

## 2. Visualisation comparative

On compare les modèles sur le macro-F1, avec leur écart-type entre sujets
(barres d'erreur) — un modèle régulier a un faible écart-type.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
colors = ['#1D9E75', '#2E75B6', '#EF9F27', '#E24B4A'][:len(results)]
ax.bar(results['Modèle'], results['Macro-F1'],
       yerr=results['Écart-type F1'], capsize=5, color=colors)
ax.set_ylabel('Macro-F1 (validation par sujet)')
ax.set_ylim(0, 1.05)
ax.set_title('Comparaison des modèles')
ax.grid(True, alpha=0.3, axis='y')
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

best = results.iloc[0]
print(f"Meilleur modèle : {best['Modèle']} (macro-F1 = {best['Macro-F1']})")

## ✅ Bilan

On dispose d'une comparaison **équitable** de plusieurs modèles : même pipeline,
même validation par sujet, mêmes métriques. Le tableau permet de justifier
objectivement le choix du modèle retenu pour l'application.

**Note :** sur de vraies données EEG, les écarts entre modèles sont plus marqués.
Un modèle avec un macro-F1 légèrement plus faible mais un écart-type plus petit
(plus régulier entre sujets) peut être préférable.